# 데이터 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd
data_path = "/content/drive/MyDrive/20250512_NLP특강/"
df = pd.read_csv(data_path + "5차년도_2차.csv", encoding="cp949")

# Label Encoding
label_map = {
    "fear": 0,
    "surprise": 1,
    "angry": 2,
    "sadness": 3,
    "neutral": 4,
    "happiness": 5,
    "disgust": 6
}
df["y"] = df["상황"].map(label_map)

x_col = '발화문'
y_col = 'y'
input_data = df[[x_col] + [y_col]]
input_data

,발화문,y
0,헐! 나 이벤트에 당첨 됐어.,5
1,내가 좋아하는 인플루언서가 이벤트를 하더라고. 그래서 그냥 신청 한번 해봤지.,5
2,"한 명 뽑는 거였는데, 그게 바로 내가 된 거야.",5
3,"당연히 마음에 드는 선물이니깐, 이벤트에 내가 신청 한번 해본 거지. 비싼 거야. ...",5
4,에피타이저 정말 좋아해. 그 것도 괜찮은 생각인 것 같애.,4
...,...,...
19369,나 엘리베이터에 갇혔어.,0
19370,하지만 기분이 나쁜 걸 어떡해?,2
19371,자취방 엘리베이턴데 정전인가봐.,0
19372,나 드디어 프로젝트 끝났어!,5


Split

In [ ]:
from sklearn.model_selection import train_test_split
trval_X, test_X, trval_y, test_y = train_test_split(
    input_data[x_col].tolist(), input_data[y_col].tolist(),
    test_size=0.05, stratify=input_data[y_col], random_state=42)

from sklearn.model_selection import train_test_split
train_X, valid_X, train_y, valid_y = train_test_split(
    trval_X, trval_y, test_size=0.05, stratify=trval_y, random_state=42)

print(f"            x      y")
print(f"train size: {len(train_X):<5}  {len(train_y):<5}")
print(f"valid size: {len(valid_X):<5}  {len(valid_y):<5}")
print(f"test size : {len(test_X):<5}  {len(test_y):<5}")

            x      y
train size: 17484  17484
valid size: 921    921  
test size : 969    969  


# 토크나이저



In [ ]:

!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.9/495.9 kB 15.2 MB/s eta 0:00:00


### 직접 구현한 커스텀 토크나이저 부분


In [ ]:

import re
import torch
from collections import defaultdict, Counter
from konlpy.tag import Okt

class CustomKoBERTTokenizer:
    def __init__(self, vocab_size=5000, special_tokens=None):
        self.vocab_size = vocab_size
        self.token_to_id = {}
        self.id_to_token = {}
        self.vocab = set()
        self.bpe_ranks = {}
        self.special_tokens = special_tokens or ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
        self.okt = Okt()

        self.spacing_correction_dict = {
            "할수있어": "할 수 있어",
            "할수있을까": "할 수 있을까",
            "볼수있어": "볼 수 있어",
            "알수있어": "알 수 있어",
            "먹을수있어": "먹을 수 있어",
            "갈수있어": "갈 수 있어",
            "올수있어": "올 수 있어",
            "못할것같아": "못 할 것 같아",
            "해야돼": "해야 돼",
            "가야돼": "가야 돼",
            "봐야돼": "봐야 돼",
            "할려고해": "하려고 해",
            "보려고해": "보려고 해",
            "하려고해": "하려고 해",
            "안돼": "안 돼",
            "됐어": "됐어",
            "안되": "안 돼",
            "안됬어": "안 됐어",
            "괜찮아": "괜찮아",
            "괜찬아": "괜찮아",
            "몰라요": "몰라요",
            "모르겠어": "모르겠어",
            "몰르겠어": "모르겠어",
            "해줄래": "해 줄래",
            "해줄수있어": "해 줄 수 있어",
            "가줄수있어": "가 줄 수 있어",
            "보여줄수있어": "보여 줄 수 있어",
            "놀러가자": "놀러 가자",
            "살수있어": "살 수 있어",
            "죽을것같아": "죽을 것 같아",
            "무서울것같아": "무서울 것 같아",
            "안좋아": "안 좋아",
            "좋아보여": "좋아 보여",
            "슬퍼보여": "슬퍼 보여",
            "웃고있어": "웃고 있어",
            "울고있어": "울고 있어",
            "가고있어": "가고 있어"
        }

    def is_reasonable(self, pos):
        return any(tag.startswith(('N', 'V', 'J', 'M')) for _, tag in pos)

    def smart_split(self, text):
        candidates = []
        for i in range(2, len(text) - 2):
            left = text[:i]
            right = text[i:]
            left_pos = self.okt.pos(left, stem=True)
            right_pos = self.okt.pos(right, stem=True)
            if self.is_reasonable(left_pos) and self.is_reasonable(right_pos):
                candidates.append((left + ' ' + right, left_pos + right_pos))

        if candidates:
            best = max(candidates, key=lambda x: len(x[1]))
            return best[0]
        return text

    def _pre_tokenize(self, text):
        for wrong, corrected in self.spacing_correction_dict.items():
            if wrong in text:
                text = text.replace(wrong, corrected)

        text = self.smart_split(text)
        morphs_with_tags = self.okt.pos(text, stem=False)
        processed = []

        common_suffixes = [
            "는다", "는다.", "했다", "합니다", "합니다.", "잤어", "잤어요",
            "었어", "았어", "네요", "네요.", "군요", "겠어", "겠어요", "지만", "아서", "어서", "해서"
        ]

        for morph, tag in morphs_with_tags:
            matched = False
            for suffix in common_suffixes:
                if morph.endswith(suffix) and len(morph) > len(suffix):
                    stem_part = morph[:-len(suffix)]
                    suffix_part = suffix
                    if tag in ['Verb', 'Adjective']:
                        processed.append(stem_part)
                        processed.append("▁")
                    else:
                        processed.append(stem_part)
                    processed.append(suffix_part)
                    matched = True
                    break

            if not matched:
                if tag in ['Verb', 'Adjective']:
                    processed.append(morph)
                    processed.append("▁")
                else:
                    processed.append(morph)

        return processed

    def _get_pairs(self, word):
        pairs = set()
        prev_char = word[0]
        for char in word[1:]:
            pairs.add((prev_char, char))
            prev_char = char
        return pairs

    def _merge(self, word, pair):
        merged = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i+1]) == pair:
                merged.append(word[i] + word[i+1])
                i += 2
            else:
                merged.append(word[i])
                i += 1
        return merged

    def train(self, corpus):
        tokenized_words = []
        for sent in corpus:
            for word in self._pre_tokenize(sent):
                tokenized_words.append(list(word) + ["</w>"])

        vocab_counter = Counter(tuple(t) for t in tokenized_words)

        for _ in range(self.vocab_size):
            pairs = defaultdict(int)
            for word, freq in vocab_counter.items():
                for pair in self._get_pairs(word):
                    pairs[pair] += freq
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            self.bpe_ranks[best] = len(self.bpe_ranks)
            new_vocab = {}
            for word, freq in vocab_counter.items():
                new_word = self._merge(word, best)
                new_vocab[tuple(new_word)] = freq
            vocab_counter = new_vocab

        final_tokens = set()
        for word in vocab_counter:
            final_tokens.update(word)
        self.vocab = list(final_tokens.union(self.special_tokens))
        self.token_to_id = {tok: i for i, tok in enumerate(self.vocab)}
        self.id_to_token = {i: tok for tok, i in self.token_to_id.items()}

    def encode(self, text):
        tokens = ['[CLS]']
        for word in self._pre_tokenize(text):
            chars = list(word) + ["</w>"]
            while True:
                pairs = self._get_pairs(chars)
                candidate = {pair: self.bpe_ranks.get(pair, float("inf")) for pair in pairs}
                if not candidate:
                    break
                best_pair = min(candidate, key=candidate.get)
                if best_pair not in self.bpe_ranks:
                    break
                chars = self._merge(chars, best_pair)
            tokens.extend(chars)
        tokens.append('[SEP]')

        ids = [self.token_to_id.get(tok, self.token_to_id["[UNK]"]) for tok in tokens]
        return tokens, ids

    def decode(self, tokens):
      cleaned = []
      for t in tokens:
          if t in self.special_tokens:
              continue
          t = t.replace("▁", "").replace("</w>", "")
          cleaned.append(t)
      return "".join(cleaned).strip()


    def __call__(self, text, padding="max_length", truncation=True, return_tensors="pt", max_length=20):
        tokens, ids = self.encode(text)
        pad_id = self.token_to_id["[PAD]"]
        if len(ids) < max_length:
            pad_len = max_length - len(ids)
            ids += [pad_id] * pad_len
            attention_mask = [1] * len(tokens) + [0] * pad_len
        else:
            ids = ids[:max_length]
            attention_mask = [1 if i != pad_id else 0 for i in ids]

        token_type_ids = [0] * max_length

        if return_tensors == "pt":
            return {
                "input_ids": torch.tensor([ids]),
                "token_type_ids": torch.tensor([token_type_ids]),
                "attention_mask": torch.tensor([attention_mask])
            }
        else:
            return {
                "input_ids": ids,
                "token_type_ids": token_type_ids,
                "attention_mask": attention_mask
            }


## vocab 생성

토크나이저 첫 생성 시 vocab 만들기 위해

문장 데이터 입력해서 학습시켜야 함

In [ ]:
corpus = input_data['발화문'].tolist()
print(len(corpus))
corpus[:3]

19374


['헐! 나 이벤트에 당첨 됐어.',
 '내가 좋아하는 인플루언서가 이벤트를 하더라고. 그래서 그냥 신청 한번 해봤지.',
 '한 명 뽑는 거였는데, 그게 바로 내가 된 거야.']

In [ ]:
custom_tokenizer = CustomKoBERTTokenizer(vocab_size=5000)
custom_tokenizer.train(corpus)

## 구현: 인코드/디코드

토크나이저 encode/decode 기능 구현되어야 함



In [ ]:
test_sentences = [
    "그는 밥을 먹는다",
    "빨리 먹는다",
    "맛있게 먹는다"
]

In [ ]:
print("=== Testing word '먹는다' ===")
for sentence in test_sentences:
    tokens, token_ids = custom_tokenizer.encode(sentence)
    print(f"\nText     : {sentence}")
    print(f"Tokens   : {tokens}")
    print(f"Token ids: {token_ids}")
    print(f"Decoded  : {custom_tokenizer.decode(tokens)}")

=== Testing word '먹는다' ===

Text     : 그는 밥을 먹는다
Tokens   : ['[CLS]', '그</w>', '는</w>', '밥</w>', '을</w>', '먹</w>', '▁</w>', '는', '다</w>', '[SEP]']
Token ids: [1110, 5390, 1750, 4929, 4769, 422, 272, 3117, 1824, 3915]
Decoded  : 그는밥을먹는다

Text     : 빨리 먹는다
Tokens   : ['[CLS]', '빨리</w>', '먹</w>', '▁</w>', '는', '다</w>', '[SEP]']
Token ids: [1110, 5058, 422, 272, 3117, 1824, 3915]
Decoded  : 빨리먹는다

Text     : 맛있게 먹는다
Tokens   : ['[CLS]', '맛있게</w>', '▁</w>', '먹</w>', '▁</w>', '는', '다</w>', '[SEP]']
Token ids: [1110, 2872, 272, 422, 272, 3117, 1824, 3915]
Decoded  : 맛있게먹는다


In [ ]:
# Test with OOV words
oov_sentences = [
    "나는 아이스아메리카노를 마신다",  # "아이스아메리카노" might be OOV
    "그는 스마트폰으로 인터넷서핑을 한다"  # "스마트폰", "인터넷서핑" might be OOV
]

print("\n=== Testing OOV handling ===")
for sentence in oov_sentences:
    tokens, token_ids = custom_tokenizer.encode(sentence)
    print(f"\nText     : {sentence}")
    print(f"Tokens   : {tokens}")
    print(f"Token ids: {token_ids}")
    print(f"Decoded  : {custom_tokenizer.decode(tokens)}")


=== Testing OOV handling ===

Text     : 나는 아이스아메리카노를 마신다
Tokens   : ['[CLS]', '나</w>', '는</w>', '아이', '스</w>', '아</w>', '메', '리</w>', '카', '노</w>', '를</w>', '마', '신', '다</w>', '▁</w>', '[SEP]']
Token ids: [1110, 724, 1750, 2615, 750, 5133, 4923, 1522, 3284, 3208, 327, 881, 4993, 1824, 272, 3915]
Decoded  : 나는아이스아메리카노를마신다

Text     : 그는 스마트폰으로 인터넷서핑을 한다
Tokens   : ['[CLS]', '그</w>', '는</w>', '스</w>', '마트</w>', '폰</w>', '으로</w>', '인터넷</w>', '서', '핑</w>', '을</w>', '한다</w>', '▁</w>', '[SEP]']
Token ids: [1110, 5390, 1750, 750, 4899, 1624, 1048, 2891, 3687, 1990, 4769, 2923, 272, 3915]
Decoded  : 그는스마트폰으로인터넷서핑을한다


## 구현: 모델 입력

In [ ]:
model_path = "drive/MyDrive/2025/KW/Model/"
model_id = "monologg/kobert"

In [ ]:
from transformers import AutoTokenizer
kobert_tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=model_path, trust_remote_code=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# 원래 토크나이저의 vocab_size와 같을 필요는 없음
kobert_tokenizer.vocab_size

8002

In [ ]:
text = "그는 밥을 먹는다"

In [ ]:
inputs = kobert_tokenizer(
            text,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
            max_length=20
        )
inputs

{'input_ids': tensor([[   2, 1191, 2266, 7088, 2010, 5760, 5782,    3,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}

In [ ]:
inputs = custom_tokenizer(
            text,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
            max_length=20
        )
inputs

{'input_ids': tensor([[1110, 5390, 1750, 4929, 4769,  422,  272, 3117, 1824, 3915,  116,  116,
           116,  116,  116,  116,  116,  116,  116,  116]]),
 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}

-> 코버트와 우리가 만든것을 비교해보는 것임

*   토큰타입아이디는 같아야됨


## 구현: 스페셜 토큰

In [ ]:
# List of the special token strings
print("All special tokens:    ", kobert_tokenizer.all_special_tokens)

# Corresponding token IDs
print("All special token IDs: ", kobert_tokenizer.all_special_ids)

# Mapping of each special token role
print("Special tokens map:    ", kobert_tokenizer.special_tokens_map)

All special tokens:     ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']
All special token IDs:  [0, 3, 1, 2, 4]
Special tokens map:     {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


똑같이 출력 시키면 됨


In [ ]:
for txt in train_X[:5]:
    token_ids = kobert_tokenizer.encode(txt)
    print(txt)
    print(" → tokens[:10]:", token_ids[:10])
    print()

그렇지. 경찰분들 진짜 고생 많으신 것 같애.
 → tokens[:10]: [2, 1205, 54, 975, 6416, 5931, 4368, 993, 6542, 517]

무서워서 잠도 못 잤어.
 → tokens[:10]: [2, 2095, 6553, 7018, 6553, 3945, 5859, 2086, 517, 0]

에이 그 정도는 아니야.
 → tokens[:10]: [2, 3289, 1185, 4099, 5760, 3097, 6844, 54, 3]

아, 진짜? 기대된다. 뭐 찾아올지.
 → tokens[:10]: [2, 3093, 46, 4368, 258, 1269, 54, 2145, 4446, 6973]

날은 아주 좋았어.
 → tokens[:10]: [2, 1407, 7086, 3128, 4208, 6855, 54, 3]



In [ ]:
for txt in train_X[:5]:
    tokens, token_ids = custom_tokenizer.encode(txt)
    print(txt)
    print(" → tokens[:10]:", tokens[:10])
    print(" → ids   [:10]:", token_ids[:10])
    print()

그렇지. 경찰분들 진짜 고생 많으신 것 같애.
 → tokens[:10]: ['[CLS]', '그렇지</w>', '▁</w>', '.</w>', '경찰</w>', '분들</w>', '진짜</w>', '고생</w>', '많</w>', '▁</w>']
 → ids   [:10]: [1110, 2796, 272, 484, 4272, 3145, 3524, 4350, 173, 272]

무서워서 잠도 못 잤어.
 → tokens[:10]: ['[CLS]', '무서워서</w>', '▁</w>', '잠</w>', '도</w>', '못</w>', '잤어</w>', '▁</w>', '.</w>', '[SEP]']
 → ids   [:10]: [1110, 3428, 272, 2817, 1760, 1725, 4716, 272, 484, 3915]

에이 그 정도는 아니야.
 → tokens[:10]: ['[CLS]', '에이</w>', '그</w>', '정</w>', '도</w>', '는</w>', '아니야</w>', '▁</w>', '.</w>', '[SEP]']
 → ids   [:10]: [1110, 4734, 5390, 1186, 1760, 1750, 375, 272, 484, 3915]

아, 진짜? 기대된다. 뭐 찾아올지.
 → tokens[:10]: ['[CLS]', '아</w>', ',</w>', '진짜</w>', '?</w>', '기대</w>', '된다</w>', '▁</w>', '.</w>', '뭐</w>']
 → ids   [:10]: [1110, 5133, 686, 3524, 4887, 688, 985, 272, 484, 2925]

날은 아주 좋았어.
 → tokens[:10]: ['[CLS]', '날</w>', '은</w>', '아주</w>', '좋</w>', '▁</w>', '았</w>', '어</w>', '.</w>', '[SEP]']
 → ids   [:10]: [1110, 1046, 1294, 4140, 2369, 272, 4540, 3490, 4

In [ ]:
!pip install transformers

In [ ]:
#import os
#data_path = "drive/MyDrive/2025/KW/Data/감정 분류를 위한 대화 음성 데이터셋/"
#os.listdir(data_path)

In [ ]:
# Seed 고정
import random
import numpy as np
import torch
import os

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # multi-gpu

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

In [ ]:
model_path = "drive/MyDrive/2025/KW/Model/"
model_id = "monologg/kobert"

In [ ]:
# 토크나이저 Train
corpus = input_data['발화문'].tolist()
custom_tokenizer = CustomKoBERTTokenizer(vocab_size=5000)
custom_tokenizer.train(corpus)

In [ ]:
print("target vocab_size:", custom_tokenizer.vocab_size)
print("actual tokens   :", len(custom_tokenizer.token_to_id))  # 실제 vocab size

target vocab_size: 5000
actual tokens   : 5442


In [ ]:
from torch.utils.data import Dataset, DataLoader

class KoBERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        inputs = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
            max_length=self.max_len
        )
        return {
            'input_ids': inputs['input_ids'].squeeze(),
            'attention_mask': inputs['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
train_dataset = KoBERTDataset(train_X, train_y, custom_tokenizer)
valid_dataset = KoBERTDataset(valid_X, valid_y, custom_tokenizer)
test_dataset = KoBERTDataset(test_X, test_y, custom_tokenizer)

In [ ]:
from torch.utils.data import Dataset, DataLoader
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from transformers import BertModel
bert = BertModel.from_pretrained(model_id, cache_dir=model_path, trust_remote_code=True)

Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]

In [ ]:
import torch.nn as nn

class KoBERTClassifier(nn.Module):
    def __init__(self, bert, num_classes, hidden_size=768, dropout=0.2):
        super(KoBERTClassifier, self).__init__()
        self.bert = bert
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output
        dropped = self.dropout(pooled)
        return self.classifier(dropped)

In [ ]:
num_classes = len(df['y'].unique())
num_classes  # 7

7

In [ ]:
model = KoBERTClassifier(bert, num_classes=num_classes).to(device)

In [ ]:
from torch.optim import AdamW
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score

# Optimizer and loss
optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

epochs = 3

for epoch in range(epochs):
    # Training
    model.train()
    train_loss = 0
    train_preds = []
    train_labels = []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} - Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

    train_loss = train_loss / len(train_loader)
    train_acc = accuracy_score(train_labels, train_preds)

    # Validation
    model.eval()
    valid_loss = 0
    valid_preds = []
    valid_labels = []

    with torch.no_grad():
        for batch in valid_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            valid_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            valid_preds.extend(preds.cpu().numpy())
            valid_labels.extend(labels.cpu().numpy())

    valid_loss = valid_loss / len(valid_loader)
    valid_acc = accuracy_score(valid_labels, valid_preds)

    print(f"Epoch {epoch+1} - loss: {train_loss:.4f}  acc: {train_acc:.4f} | "
          f"val loss: {valid_loss:.4f}  val acc: {valid_acc:.4f}")

Epoch 1 - Training:   0%|          | 0/274 [00:00<?, ?it/s]

In [ ]:
# Evaluation
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {acc:.4f}")

In [ ]:
# ✅ 1) 욕설 단어 분리
"""
class CustomInsultSplitterPreTokenizer:
    def apply(self, text):
        insults = ['또라이', '새끼', '지랄']
        for word in insults:
            text = text.replace(word, f' {word} ')
        return text

# ✅ 2) 강조 접두사 분리
class CustomEmphasisPreTokenizer:
    def apply(self, text):
        prefix = ['개','존','핵','겁','겁나','졸','엄청','되게','아주','진짜','완전','정말']
        core = ['좋아','좋다','재밌','예쁘','멋지','무섭','귀엽','싫','슬프','기쁘','감동','편하','불편','놀랍','졸리']
        p2 = r'\b(' + '|'.join(prefix) + r')(' + '|'.join(prefix) + r')(' + '|'.join(core) + r'[가-힣]*)'
        p1 = r'\b(' + '|'.join(prefix) + r')(' + '|'.join(core) + r'[가-힣]*)'
        text = re.sub(p2, r'\1 \2 \3', text)
        text = re.sub(p1, r'\1 \2', text)
        return text

# ✅ 3) 강조 접두사 붙임 오류 교정
class CustomSpacingCorrectionPreTokenizer:
    def apply(self, text):
        emphasis = ['완전','진짜','정말','되게','엄청','아주','너무','개','존','핵']
        semantics = ['좋아','좋다','재밌','예쁘','멋지','무섭','귀엽','싫','슬프','기쁘',
                     '감동','편하','불편','놀랍','졸리','깨끗','시끄럽','복잡','간단',
                     '행복','배고프','배부르','피곤','따뜻','추워','덥','아프','편안','힘들']
        p2 = r'\b(' + '|'.join(emphasis) + r')(' + '|'.join(emphasis) + r')(' + '|'.join(semantics) + r'[가-힣]*)'
        p1 = r'\b(' + '|'.join(emphasis) + r')(' + '|'.join(semantics) + r'[가-힣]*)'
        text = re.sub(p2, r'\1 \2 \3', text)
        text = re.sub(p1, r'\1 \2', text)
        p_general = r'\b(' + '|'.join(emphasis) + r')([가-힣]{2,})'
        text = re.sub(p_general, r'\1 \2', text)
        return text

# ✅ 4) 부정어 분리
class CustomNegationPreTokenizer:
    def apply(self, text):
        text = re.sub(r'(안|못)([가-힣]+)', r'\1 \2', text)
        return text

# ✅ 5) 복합 조사 분리
class CustomParticleSplitterPreTokenizer:
    def apply(self, text):
        particles = ['까지', '마저', '조차', '조금', '라도', '조금이라도']
        for p in particles:
            text = re.sub(rf'{p}도', f'{p} 도', text)
            text = re.sub(rf'{p}만', f'{p} 만', text)
        return text

# ✅ 6) 명사 + 조사, 명사 + 해 분리
class CustomNounParticleEndingSplitterPreTokenizer:
    def apply(self, text):
        # 명사 + 해 (예: 우울해 → 우울 해)
        text = re.sub(r'([가-힣]+)(해)', r'\1 \2', text)

        # 명사 + 조사
        particles = [
            '이', '가', '은', '는', '을', '를',
            '에', '에서', '로', '으로', '만', '도',
            '까지', '부터', '에게'
        ]
        p = '|'.join(sorted(particles, key=len, reverse=True))
        text = re.sub(rf'([가-힣]+)({p})\b', r'\1 \2', text)
        return text

# ✅ 7) 복합 보조동사 + 복합 어미 연쇄 분리
class CustomVerbComplexSplitterPreTokenizer:
    def __init__(self, repeat=3):
        self.repeat = repeat
        self.auxiliaries = ['보', '봤', '버리', '두', '놓', '주', '가', '오', '내', '대', '드리']
        self.endings = [
            '더라고', '더라', '더래', '더니', '길래', '겠네', '겠어', '겠지',
            '었지', '었네', '었어', '었는데', '었을', '었을까', '었겠', '었',
            '으면', '으니까', '으니', '게', '고', '지', '어', '아', '래', '라', '니', '네'
        ]
        self.aux_pattern = '|'.join(sorted(self.auxiliaries, key=len, reverse=True))
        self.end_pattern = '|'.join(sorted(self.endings, key=len, reverse=True))

    def apply(self, text):
        for _ in range(self.repeat):
            text = re.sub(rf'([가-힣]+)({self.aux_pattern})', r'\1 \2', text)
            text = re.sub(rf'([가-힣]+)({self.end_pattern})\b', r'\1 \2', text)
        return text

# ✅ 8) 감탄사 & 반복 문자 정규화
class CustomExclamationPreTokenizer:
    def apply(self, text):
        exclamations = ['헐', '와', '우와', '아', '야', '어머', '으악', '앗', '오']
        for word in exclamations:
            text = re.sub(rf'({word})', r' \1 ', text)
        text = re.sub(r'(ㅋ){2,}', 'ㅋㅋ', text)
        text = re.sub(r'(ㅎ){2,}', 'ㅎㅎ', text)
        text = re.sub(r'([!?.])\1+', r'\1', text)
        return text

# ✅ 9) 구두점 공백 처리
class CustomPunctuationPreTokenizer:
    def apply(self, text):
        text = re.sub(r'([!?]{2,})', r' \1 ', text)
        text = re.sub(r'(\.{2,})', lambda m: ' '.join(list(m.group(1))), text)
        text = re.sub(r'([.,!?])', r' \1 ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

# ✅ 10) 숫자 처리
class CustomNumberUnitSplitterPreTokenizer:
    def apply(self, text):
        # 1) 아라비아 숫자 + 한글 단위 → split
        text = re.sub(r'(\d+)([가-힣]+)', r'\1 \2', text)

        # 2) 한글 숫자 + 단위 (예: 한번, 두번, 세번, 네번)
        korean_numbers = ['한', '두', '세', '네', '열', '스무']
        units = ['번', '명', '개', '시', '시간', '살', '층', '명분', '권']  # 원하는 단위 추가
        p_num = '|'.join(korean_numbers)
        p_unit = '|'.join(units)
        text = re.sub(rf'({p_num})({p_unit})', r'\1 \2', text)

        return text


# ✅ 11) 형태소 분석기 (Okt)
class MorphAnalyzerPreTokenizer:
    def __init__(self):
        self.okt = Okt()

    def apply(self, text):
        morphs = self.okt.morphs(text)
        return ' '.join(morphs)

def apply_all_custom_pretokenizers(text):
    for processor in [
        CustomInsultSplitterPreTokenizer(),
        CustomEmphasisPreTokenizer(),
        CustomSpacingCorrectionPreTokenizer(),
        CustomNegationPreTokenizer(),
        CustomParticleSplitterPreTokenizer(),
        CustomNounParticleEndingSplitterPreTokenizer(),
        CustomNumberUnitSplitterPreTokenizer(),  # ✅ 숫자+단위 규칙
        CustomVerbComplexSplitterPreTokenizer(),
        CustomExclamationPreTokenizer(),
        CustomPunctuationPreTokenizer(),
        MorphAnalyzerPreTokenizer(),
    ]:
        text = processor.apply(text)
    return text

1. 욕설/비속어 사전 분리
2. 강조 접두사(개, 존, 핵 등) + 감정 코어 분리
3. 강조 접두사 붙임 오류 교정 — 중복 접두사 + 일반 단어 뭉침 자동 교정
4. 부정어 안, 못 이 붙은 걸 분리
5. 일부 복합 조사 (조차도, 까지도, 라도만 등) 분리
6. 해 분리 (우울해 → 우울 해)
7. 복합 보조동사/복합 어미 연쇄를 반복적으로 분리
8. 감탄사 단독 분리 + 웃음/반복 문자 정규화
9. 구두점(!, ., ?, ,)에 공백 넣고 연속 기호 정리
10. 숫자 + 단위 (한시간 → 한 시간)

이후 형태소 분석기

    """
